### Cohort Analysis
If a customer gets a delayed package, do they ever come back to buy from us again? Or do we lose them to our competitors forever?"

In [ ]:
# import required libraries
import numpy as np
import pandas as pd

In [ ]:
# import orders dataset
orders_date_cols = ["order_purchase_timestamp",
                    "order_approved_at",
                    "order_delivered_carrier_date",
                    "order_delivered_customer_date",
                    "order_estimated_delivery_date"]
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv", parse_dates=orders_date_cols)

# import customers datasets
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

In [ ]:
# Merge datasets
df = pd.merge(orders, customers, on="customer_id")

In [ ]:
df.sample(5)

In [ ]:
df.sort_values(by=["customer_unique_id", "order_purchase_timestamp"], inplace=True)

In [ ]:
first_orders = df.drop_duplicates(subset=["customer_unique_id"], keep="first").copy()

In [ ]:
first_orders = first_orders.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date'])

In [ ]:
first_orders["delivery_delay_days"] = (first_orders["order_delivered_customer_date"] - first_orders["order_estimated_delivery_date"]).dt.days

In [ ]:
first_orders["is_delayed"] = first_orders["delivery_delay_days"] > 0

In [ ]:
total_orders_per_customer = df.groupby('customer_unique_id')['order_id'].nunique().reset_index()
total_orders_per_customer.rename(columns={'order_id': 'total_orders'}, inplace=True)

first_orders = first_orders.merge(total_orders_per_customer, on='customer_unique_id', how='left')

In [ ]:
first_orders['made_repeat_purchase'] = first_orders['total_orders'] > 1

In [ ]:
# Group by whether the first order was delayed
repeat_rates = first_orders.groupby('is_delayed')['made_repeat_purchase'].mean() * 100
repeat_rates

### Interpretation
Customers whose first order was delivered on time return at a higher rate (e.g., 3.11 %) than those who experienced a delay (e.g., 2.56 %).
The first delivery experience does appear to influence the likelihood of a repeat purchase.
